# FloodTrack B — Colab CDSE Runbook

**No Google Earth Engine.** Uses **free Copernicus Data Space** public STAC + Sentinel Hub Processing API, and historical OSM.

**Run in order.** First smoke-test without credentials, then add CDSE secrets, discover a valid same-orbit pair, acquire once, rerun staged, optionally train AI, and export the **entire project ZIP**.

Read `COLAB_STEP_BY_STEP.md` for full explanations. **This ZIP contains synthetic examples only; no real satellite data or trained weights are included until you acquire/train them.**

## 0 — Platform setup
Open Colab; upload the ZIP using Files. Set T4 GPU only if training.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
zip_path=Path('/content/FloodTrack_B_Full_Project_CDSE.zip')
assert zip_path.exists(), 'Upload the project ZIP to /content in Colab Files first'
with ZipFile(zip_path) as z: z.extractall('/content')
ROOT=Path('/content/FloodTrack_B_Complete')
assert (ROOT/'README.md').exists()
print('Project:',ROOT)

## 1 — Install dependencies and verify offline tests

In [ ]:
%cd /content/FloodTrack_B_Complete
%pip install -q -r requirements.txt
!python -m pytest -q tests
!python scripts/run_pipeline.py --config config/smoke_test.yaml

In [ ]:
from IPython.display import display,IFrame
display(IFrame('outputs/synthetic_training_and_pipeline_smoke/dashboard_map.html',width='100%',height=600))

## 2 — Load secrets
Create **CDSE_CLIENT_ID** and **CDSE_CLIENT_SECRET** in the platform secret manager first. These come from your Sentinel Hub OAuth client; never print their values.

In [ ]:
import os
from google.colab import userdata
os.environ['CDSE_CLIENT_ID']=userdata.get('CDSE_CLIENT_ID')
os.environ['CDSE_CLIENT_SECRET']=userdata.get('CDSE_CLIENT_SECRET')
assert os.environ['CDSE_CLIENT_ID'] and os.environ['CDSE_CLIENT_SECRET']
print('CDSE credentials loaded (values not printed)')

In [ ]:
!python scripts/run_pipeline.py --config config/case_study.yaml --cdse-check

## 3 — Configure example real AOI
This bbox is **illustrative**. Confirm your target area and event date. The historical OSM date must be strictly pre-event.

In [ ]:
import yaml
from pathlib import Path
cfg_path=Path('config/case_study.yaml')
cfg=yaml.safe_load(cfg_path.read_text())
cfg.update({'name':'trishuli_2026_demo_aoi', 'source_mode':'real',
            'case_event_date':'2026-08-26', 'osm_snapshot':'2026-07-27',
            'bbox':[85.119,28.150,85.157,28.182], 'scale_m':30,
            'lookback_days':30, 'lookforward_days':25,
            'use_optical':True, 'use_dem':True, 'ai_model_path':None})
cfg_path.write_text(yaml.safe_dump(cfg,sort_keys=False))
print('Saved config:',cfg_path)

## 4 — Public STAC preflight
Fails safely if no same relative orbit and pass before/after the event. No credentials needed for this metadata query.

In [ ]:
!python scripts/check_cdse_catalog.py --config config/case_study.yaml

## 5 — Download real CDSE satellite GeoTIFFs and historical OSM
Uses CDSE free-tier processing quota. This step requires Internet and valid OAuth client credentials. No official damage/reference maps are used.

In [ ]:
!python scripts/download_inputs.py --config config/case_study.yaml

### If ohsome returns HTTP 403 (recover without redownloading satellite images)
The historical snapshot endpoint is affected by a reported server-side 403.
The project automatically tries the full-history geometry endpoint and selects
only versions active on the configured pre-event date. If the original download
cell failed, rerun **only the OSM acquisition** in the next cell. Do not use
present-day Overpass or OSM bounding boxes as a substitute.


In [ ]:
!python scripts/download_inputs.py --config config/case_study.yaml --osm-only


## 6 — Reuse downloaded inputs in staged mode
This does not redownload data. The `name` must match the acquisition.

In [ ]:
src=yaml.safe_load(Path('config/case_study.yaml').read_text())
src['source_mode']='staged'
Path('config/staged_example.yaml').write_text(yaml.safe_dump(src,sort_keys=False))
!python scripts/run_pipeline.py --config config/staged_example.yaml

In [ ]:
out=Path('outputs')/src['name']
print((out/'impact_metrics.json').read_text()[:2500])
display(IFrame(str(out/'dashboard_map.html'),width='100%',height=650))

## 7 — OPTIONAL real-data AI training (requires external approved dataset)
Download Kuro Siwo / Sen1Floods11 according to its own README. Inspect license and class codes; adapt to `data/training/pairs/SITE_ID/{pre_vv,post_vv,flood_label}.tif` with VV in dB. **Do not run until these real files exist and label mapping is verified.**

In [ ]:
# Uncomment ONLY after inspecting the dataset and preparing real pairs:
# !python scripts/prepare_training.py --pairs data/training/pairs --positive-values 1 --negative-values 0
# !python scripts/train_segmentation.py --patches data/training/patches --epochs 10 --batch 8 --val-sites SITE_HELD_OUT
# src['ai_model_path']='models/unet_s1.pt'
# Path('config/staged_example.yaml').write_text(yaml.safe_dump(src,sort_keys=False))
# !python scripts/run_pipeline.py --config config/staged_example.yaml

## 8 — Validate and prepare deliverables
Inspect masks, road-network unknowns and limitations. Freeze predictions before evaluating against EMSR927. Complete `templates/` report/video/checklist. See `PHASE_PLAN.md`.

## 9 — ZIP the **entire** working project (not just outputs)
Includes acquired data and model weights if present, source code, configs, notebooks, test files and outputs. Excludes common secret files.

In [ ]:
!python scripts/package_everything.py --out /content/FloodTrack_B_All_Files.zip
from zipfile import ZipFile
zip_path=Path('/content/FloodTrack_B_All_Files.zip')
with ZipFile(zip_path) as z:
    names=z.namelist()
    print('Archive files:',len(names),'Bytes:',zip_path.stat().st_size)
    print('Has code:',any('src/floodtrack/acquisition.py' in x for x in names))
    print('Has outputs:',any('/outputs/' in x for x in names))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import shutil
shutil.copy2('/content/FloodTrack_B_All_Files.zip','/content/drive/MyDrive/FloodTrack_B_All_Files.zip')
print('Copied ZIP to Drive')

### Security and scientific honesty
Do not publish OAuth secrets, synthetic outputs as Nepal observations, reference EMS maps as model inputs, or unverified damage claims. Live API behavior/quotas can change. The pipeline is small-AOI and must be extended for arbitrarily large areas.